In [1]:
import numpy as np
import time
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp

# ==========================================
# 1. LOAD DATA (Concrete ID 4353 - 8 Features)
# ==========================================
print("Fetching Concrete Data...")
concrete = fetch_openml(data_id=4353, as_frame=True, parser='auto')

if concrete.target is not None:
    X_all = concrete.data.to_numpy(dtype=float)
    y_all = concrete.target.to_numpy(dtype=float)
else:
    full_data = concrete.data.to_numpy(dtype=float)
    X_all = full_data[:, :-1]
    y_all = full_data[:, -1]

# Scale features to [-π, π] and standardize target
scaler_X = MinMaxScaler(feature_range=(-np.pi, np.pi))
X_scaled = scaler_X.fit_transform(X_all)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y_all.reshape(-1, 1)).flatten()

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

# ==========================================
# 2. THE 4-QUBIT / 4-LAYER ENGINE (48 Params)
# ==========================================
n_qubits = 4
num_features = 8  
num_layers = 4    # UPGRADED TO 4 LAYERS!
total_q_params = n_qubits * 3 * num_layers # 48 params

x_params = ParameterVector('x', num_features)
theta_params = ParameterVector('θ', total_q_params)

# Dense Feature Map (2 features per qubit)
feature_map = QuantumCircuit(n_qubits)
for i in range(n_qubits):
    feature_map.rx(x_params[2 * i], i)
    feature_map.ry(x_params[2 * i + 1], i)

# Deep Ansatz (4 Layers of Entanglement)
ansatz = QuantumCircuit(n_qubits)
for layer in range(num_layers):
    offset = layer * (n_qubits * 3) 
    for i in range(n_qubits):
        ansatz.rz(theta_params[offset + i], i)
        ansatz.ry(theta_params[offset + n_qubits + i], i)
        ansatz.rz(theta_params[offset + (2*n_qubits) + i], i)
    
    # Entanglement Ring
    for i in range(n_qubits - 1):
        ansatz.cx(i, i + 1)
    ansatz.cx(n_qubits - 1, 0)

full_circuit = feature_map.compose(ansatz)
print(f"✅ 4-Qubit, 4-Layer circuit built with {total_q_params} trainable parameters.")

observables = [SparsePauliOp.from_sparse_list([("Z", [i], 1)], num_qubits=n_qubits) for i in range(n_qubits)]
estimator = StatevectorEstimator()

def get_quantum_embedding(x_data, theta_weights):
    params = list(x_data) + list(theta_weights)
    pub = (full_circuit, observables, params)
    job = estimator.run([pub])
    return np.array(job.result()[0].data.evs)

def get_quantum_gradients(x_data, theta_weights):
    shift = np.pi / 2
    num_weights = len(theta_weights) 
    jacobian = np.zeros((n_qubits, num_weights)) # 4x48 matrix
    
    for i in range(num_weights):
        theta_plus = theta_weights.copy()
        theta_plus[i] += shift
        emb_plus = get_quantum_embedding(x_data, theta_plus)
        
        theta_minus = theta_weights.copy()
        theta_minus[i] -= shift
        emb_minus = get_quantum_embedding(x_data, theta_minus)
        
        jacobian[:, i] = 0.5 * (emb_plus - emb_minus)
        
    return jacobian

Fetching Concrete Data...
✅ 4-Qubit, 4-Layer circuit built with 48 trainable parameters.


In [2]:
# ==========================================
# 3. TRAINING LOOP & TESTING (Regression)
# ==========================================
np.random.seed(42)

# Initialize Weights (53 Total)
theta_weights = np.random.uniform(-np.pi, np.pi, total_q_params)  
c_weights = np.random.randn(n_qubits) # 4 classical weights
c_bias = np.random.randn(1)[0]        # 1 classical bias

learning_rate = 0.01
epochs = 5  
total_train_samples = len(X_train) 

print("\n" + "=" * 50)
print(f"🏗️ STARTING 4-LAYER DEEP QUANTUM REGRESSION")
print(f"Total Parameters: 53 (48 Quantum + 5 Classical)")
print("=" * 50)

start_time = time.time()

for epoch in range(epochs):
    total_loss = 0
    indices = np.arange(len(X_train))
    np.random.shuffle(indices) # Mix them up!
    for i in indices: 
        x_data = X_train[i] 
        true_strength = y_train[i]
        
        # --- FORWARD PASS ---
        embedding = get_quantum_embedding(x_data, theta_weights)
        prediction = np.dot(c_weights, embedding) + c_bias
        
        # Mean Squared Error (MSE)
        error = prediction - true_strength
        total_loss += 0.5 * (error ** 2)
        
        # --- BACKWARD PASS ---
        d_prediction = error  
        d_c_weights = d_prediction * embedding
        d_c_bias = d_prediction
        d_embedding = d_prediction * c_weights
        
        # Parameter-Shift Gradients (48 shifts per row!)
        jacobian = get_quantum_gradients(x_data, theta_weights)
        d_theta = np.dot(d_embedding, jacobian)
        
        # --- UPDATE WEIGHTS ---
        c_weights -= learning_rate * d_c_weights
        c_bias -= learning_rate * d_c_bias
        theta_weights -= learning_rate * d_theta
        
        if (i + 1) % 50 == 0:
            print(".", end="", flush=True)

    learning_rate *= 0.8 
    print(f"New LR: {learning_rate:.4f}")

    avg_loss = total_loss / total_train_samples
    print(f" | Epoch {epoch+1}/{epochs} | Avg MSE Loss: {avg_loss:.4f}")

print(f"\n✅ Training Complete! Time: {(time.time() - start_time)/60:.2f} minutes.")

# ==========================================
# 4. FINAL EVALUATION ON UNSEEN DATA
# ==========================================
total_absolute_error = 0

for i in range(len(X_test)):
    x_data = X_test[i]
    true_strength = y_test[i]
    
    embedding = get_quantum_embedding(x_data, theta_weights)
    prediction = np.dot(c_weights, embedding) + c_bias
    total_absolute_error += np.abs(prediction - true_strength)

mae = total_absolute_error / len(X_test)
print("-" * 50)
print(f"🏆 Final Test MAE: {mae:.4f}")
print("-" * 50)


🏗️ STARTING 4-LAYER DEEP QUANTUM REGRESSION
Total Parameters: 53 (48 Quantum + 5 Classical)
................New LR: 0.0080
 | Epoch 1/5 | Avg MSE Loss: 0.4305
................New LR: 0.0064
 | Epoch 2/5 | Avg MSE Loss: 0.3318
................New LR: 0.0051
 | Epoch 3/5 | Avg MSE Loss: 0.2919
................New LR: 0.0041
 | Epoch 4/5 | Avg MSE Loss: 0.2736
................New LR: 0.0033
 | Epoch 5/5 | Avg MSE Loss: 0.2634

✅ Training Complete! Time: 56.06 minutes.
--------------------------------------------------
🏆 Final Test MAE: 0.6010
--------------------------------------------------


In [ ]:
import numpy as np
import time
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp

# ==========================================
# 1. LOAD DATA (Concrete ID 4353 - 8 Features)
# ==========================================
print("Fetching Concrete Data...")
concrete = fetch_openml(data_id=4353, as_frame=True, parser='auto')

if concrete.target is not None:
    X_all = concrete.data.to_numpy(dtype=float)
    y_all = concrete.target.to_numpy(dtype=float)
else:
    full_data = concrete.data.to_numpy(dtype=float)
    X_all = full_data[:, :-1]
    y_all = full_data[:, -1]

# Scale features to [-π, π] and standardize target
scaler_X = MinMaxScaler(feature_range=(-np.pi, np.pi))
X_scaled = scaler_X.fit_transform(X_all)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y_all.reshape(-1, 1)).flatten()

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

Fetching Concrete Data...


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

# 1. THE 53-PARAMETER NEURAL NETWORK
# Architecture: 8 inputs -> 4 neurons -> 3 neurons -> 1 output
# (8*4)+4 + (4*3)+3 + (3*1)+1 = 36 + 15 + 4 = 55 parameters (closest possible match)
mlp_restricted = MLPRegressor(hidden_layer_sizes=(4, 3), 
                               activation='tanh', 
                               solver='lbfgs', 
                               max_iter=2000, 
                               random_state=42)

mlp_restricted.fit(X_train, y_train)
mlp_mae = mean_absolute_error(y_test, mlp_restricted.predict(X_test))

# 2. THE 53-PARAMETER POLYNOMIAL MODEL
# A 2nd-degree polynomial of 8 features creates exactly 45 coefficients + 8 inputs = 53
poly_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=True)),
    ("linear", LinearRegression())
])

poly_model.fit(X_train, y_train)
poly_mae = mean_absolute_error(y_test, poly_model.predict(X_test))

print("="*50)
print("⚖️ FAIR COMPARISON (Parameter Budget: ~53)")
print("="*50)
print(f"Classical Neural Network MAE: {mlp_mae:.4f}")
print(f"Polynomial Regressor MAE:     {poly_mae:.4f}")
print(f"Deep Quantum Circuit MAE:     [Waiting for your training...]")
print("="*50)

⚖️ FAIR COMPARISON (Parameter Budget: ~53)
Classical Neural Network MAE: 0.2768
Polynomial Regressor MAE:     0.3575
Deep Quantum Circuit MAE:     [Waiting for your training...]


/home/sriram/Sriram/SEM 8/BTP/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

# 1. RESTRICTED NEURAL NETWORK (Approx 53 Params)
# (8 in * 4 hidden) + 4 bias + (4 hidden * 3 hidden) + 3 bias + (3 hidden * 1 out) + 1 bias = 55
mlp_small = MLPRegressor(hidden_layer_sizes=(4, 3), activation='tanh', 
                         solver='lbfgs', max_iter=2000, random_state=42)
mlp_small.fit(X_train, y_train)
mlp_mae = mean_absolute_error(y_test, mlp_small.predict(X_test))

# 2. RESTRICTED DECISION TREE
# A tree with 26 "splits" has roughly 53 parameters (thresholds and leaf values)
tree_small = DecisionTreeRegressor(max_leaf_nodes=27, random_state=42)
tree_small.fit(X_train, y_train)
tree_mae = mean_absolute_error(y_test, tree_small.predict(X_test))

# 3. POLYNOMIAL REGRESSION (Degree 2)
# 8 features expanded to 2nd degree creates 45 terms + intercept = 46 parameters
# This is the closest classical "math-heavy" equivalent to a quantum feature map.
poly_small = Pipeline([
    ("poly", PolynomialFeatures(degree=2)),
    ("ridge", Ridge(alpha=1.0))
])
poly_small.fit(X_train, y_train)
poly_mae = mean_absolute_error(y_test, poly_small.predict(X_test))

print("=" * 60)
print(f"{'MODEL':<30} | {'MAE (Lower is Better)':<20}")
print("-" * 60)
print(f"{'Classical Neural Network':<30} | {mlp_mae:.4f}")
print(f"{'Small Decision Tree':<30} | {tree_mae:.4f}")
print(f"{'Polynomial Regressor':<30} | {poly_mae:.4f}")
print(f"{'DEEP QUANTUM CIRCUIT (4-Layer)':<30} | [WAITING...]")
print("=" * 60)

MODEL                          | MAE (Lower is Better)
------------------------------------------------------------
Classical Neural Network       | 0.2768
Small Decision Tree            | 0.4145
Polynomial Regressor           | 0.3553
DEEP QUANTUM CIRCUIT (4-Layer) | [WAITING...]


/home/sriram/Sriram/SEM 8/BTP/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
